# Datamine LAYOUT Process Example

This notebook demonstrates how to configure and run the **`layout`** process wrapper in `dmstudio`.

## Process Description

## LAYOUT

Lay out a blast pattern.

You pick a blast from the input file to work on; this defines both the Bench and Blast numbers which will remain in force until a new blast is chosen.

If the blast specified is not on file (for example, blast 0) then all holes on the input file will be read, and a dummy rectangular outline constructed around them. The program automatically works out the co-ordinate limits of this blast, and will compute a rotation angle to align the major axis of the blast along the screen X axis; the user may override this if required. The blast is then shown on the graphics screen.
A set of grid lines in the original co-ordinate system is also shown, rotated if the blast was rotated. Different colours may be chosen (under user control) for the two axes, to give an indication of which axis is which. A suitable grid size is chosen automatically, but the user may change or suppress this if required (DEFAULTS [/CF] command).

If there are any existing blast-holes on file, these may be read in (GET HOLES [/FO] command) and displayed. A set of colours is automatically chosen to display blast-hole grades in, based upon the first grade field found in the blast-hole file (if any). The user may choose different fields, colours, grade ranges etc. by use of the BLASTHOLE DISPLAY [/CD] command.

A title is automatically generated giving the Bench and Blast numbers; again, this may be substituted by use of the TITLE [/DT] command.

### Input Files:

* **in** (String):
  Blast outline for composite design. The file must contain the fields:-
  XP,YP,ZP,PTN,PVALUE (numeric, explicit). It may also contain the field BENCH; if it does
  not contain this field, all bench numbers will be taken as 1\. The PVALUE field contains
  a Blast number. The file may contain many blasts; one is selected at the start of the
  process.
  Required=Yes

* **text** (Undefined):
  Input/output text file for text added to the drawing. If this file does not exist, it
  will be created. If it does exist, it must have the fields BENCH, BLAST, COMPOSIT, XT,
  YT, ANGLE, CHARSIZE, ASPRAT, (all numeric) and TEXT (40 characters). Any existing text
  in the file for the current bench and blast will be plotted on the screen.
  Required=Yes

* **outlines** (String):
  Input/output perimeters. If this file does not exist, it will be created. It must
  contain the fields XP, YP, ZP, PTN, PVALUE, P, PTYPE, BENCH, BLAST. The PVALUE field
  will contain the perimeter number. Any perimeter defined with the DIGITISER [/CD] or NEW
  PERIMETER [/EN] commands may be written to the OUTLINES file by the WRITE PERIMETER
  [/FW] command. Perimeters will be overwritten if they match the perimeter number
  (PVALUE) of the perimeter being written.
  Required=Yes

* **patterns** (Undefined):
  Input/output pattern file. If this file does not exist, it will be created. It must
  contain the fields ROW, XS, XSPACING, YSPACING, PATTERN (all numeric) and PATTEXT (16
  character alphanumeric).
  Required=Yes

* **collars** (Undefined):
  Input/output collars file. Fields required are XCOLLAR, YCOLLAR, ZCOLLAR, BENCH, BLAST
  and BHID (A/N). Additional fields used if available are BRG, DIP, HLENGTH, PATTERN,
  NSAMP and SNFIRST. If this file does not exist it will be created with all the above
  fields. At least one of the COLLARS or HOLES files must be specified.
  Required=No

* **holes** (Undefined):
  Blast hole samples file. Fields required are X, Y, Z and BHID (A/N). Additional fields
  used if available are BENCH, BLAST, A0, B0, LENGTH, SAMPLE, FROM and TO. If this file
  does not exist it will be created with all the above fields. If it contains any grade
  values, these may be displayed either numerically or by colour besides each blast-hole.
  This file will be written to by the WRITE BLAST HOLES [/FL] command. If any entries
  exist on the file for the current Bench and Blast they will first be overwritten. If the
  BENCH and BLAST fields do not exist, then all entries will be deleted before the new
  holes are written. At least one of the COLLARS or HOLES files must be specified.
  Required=No

* **geol** (String):
  Geological boundaries. This file must contain the fields X,Y,Z,PTN and PVALUE. The
  values are assumed to be (unclosed) strings rather than perimeters. Any strings on this
  file may be plotted over the blast.
  Required=No

### Output Files:

### Fields:

### Parameters:

* **coordmod**:
  Coordinate verification mode, controls the prompting for coordinates in the LAY DOWN
  PATTERN [/BP] and DEFINE HOLE [/BD] commands. Options: 0: No coordinate prompting; 1:
  Coordinates are prompted for.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **coordtyp**:
  Coordinate type: Options: 0: Conventional rhs; 1: LO co-ordinate system.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **loyorig**:
  For **COORDTYP** =1 only; the LO Y co-ordinate origin [including \- sign] for internal
  co-ordinate conversion.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **loxorig**:
  For **COORDTYP** =1 only; the LO X co-ordinate origin for internal co-ordinate
  conversion.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **haxiscol**:
  Colour for horizontal axis lines; these are X axis lines [ **COORDTYP** =0] or LO Y
  lines [ **COORDTYP** =1] (8).
  Range=1,64
  Values=Undefined
  Default=8
  Required=No

* **vaxiscol**:
  Colour for vertical axis lines; these are Y axis lines [ **COORDTYP** =0] or LO X lines

## [ **COORDTYP** =1] (10).

  Range=1,64
  Values=Undefined
  Default=10
  Required=No

* **charsize**:
  Character size for display in mm (3).
  Range=Undefined
  Values=Undefined
  Default=3
  Required=No

* **dimenu**:
  Toggle between cursor and digitiser mode. Options: 0: Cursor mode.; 1: Digitiser mode.
  All commands available from digitiser. Default is 0
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('layout')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute layout
print("Running layout...")
dm_cmd.layout(
    in_i='t_assays',  # required
    text_i='t_input_file',  # required
    outlines_i=['t_input_file'],  # required
    patterns_i=['t_input_file'],  # required
    # collars_i=['t_collars'],  # optional
    # holes_i='optional',  # optional
    # geol_i='optional',  # optional
    # coordmod_p=0,  # optional
    # coordtyp_p=0,  # optional
    # loyorig_p='optional',  # optional
    # loxorig_p='optional',  # optional
    # haxiscol_p=8,  # optional
    # vaxiscol_p=10,  # optional
    # charsize_p=3,  # optional
    # dimenu_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("layout execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_layout_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")